# 🚀 NB03：進階 RAG — 混合搜索與重排序

**目標：** 掌握兩個關鍵的進階 RAG 技術，大幅提升檢索品質。

**學習成果：**
- 理解純向量搜索的盲點，以及 BM25 的互補優勢
- 實作混合搜索（Hybrid Search）：BM25 + 向量搜索結合
- 掌握 Reciprocal Rank Fusion（RRF）排名融合算法
- 理解 Reranking（重排序）的原理，用 Cross-Encoder 實作

> 💡 **一句話說重點：** 向量搜索擅長語意，BM25 擅長關鍵字精確匹配；混合搜索兩者兼得，Reranking 再精煉結果。

## 1. 純向量搜索的盲點

### 問題場景

假設你的文件庫有：
- 文件 A：「Python 程式語言由 Guido van Rossum 在 1991 年創建」
- 文件 B：「蟒蛇（Python）是分布在東南亞的大型爬蟲類...」

查詢：「Python 的創建者是誰？」

**向量搜索的問題：**
- 向量搜索基於語意相似度，「Python 程式語言」和「蟒蛇」在向量空間中可能距離不遠（同一個詞）
- 當查詢包含**專有名詞、縮寫、版本號**（如 GPT-4、React 18、BERT）時，向量搜索可能找錯文件

**BM25 的優勢：**
- 基於關鍵字頻率（TF-IDF 變體），完全匹配關鍵字時表現極佳
- 對精確詞彙匹配更可靠

```
向量搜索 (Dense)  ←→  BM25 (Sparse)
                混合搜索
        兩者的優點取聯集！
```

## 2. 環境設定

In [ ]:
import os
import math
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI
import chromadb
from rank_bm25 import BM25Okapi

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
chroma_client = chromadb.Client()

print("✅ 環境設定完成")

## 3. 準備測試資料

設計一組文件，包含容易讓純向量搜索失誤的案例（專有名詞、技術術語）。

In [ ]:
# 知識庫文件
documents = [
    {
        "id": "d01",
        "content": "BERT（Bidirectional Encoder Representations from Transformers）是 Google 在 2018 年發布的預訓練語言模型。BERT 使用雙向 Transformer 編碼器，能夠同時理解句子前後文。BERT 的預訓練任務包括遮罩語言模型（MLM）和下一句預測（NSP）。在 GLUE 基準測試上，BERT 創下了 80.5% 的高分。"
    },
    {
        "id": "d02",
        "content": "GPT-4 是 OpenAI 在 2023 年 3 月發布的大型語言模型，參數量估計超過 1 兆。GPT-4 是多模態模型，能處理文字和圖片輸入。在律師資格考試中，GPT-4 達到前 10% 的成績。GPT-4 Turbo 版本具有更長的 128K token 上下文窗口。"
    },
    {
        "id": "d03",
        "content": "RAG（Retrieval-Augmented Generation）架構結合了信息檢索和文字生成。RAG 系統包含兩個核心元件：Retriever（檢索器）負責從知識庫找到相關文件，Generator（生成器）根據檢索結果生成回答。RAG 能有效減少 LLM 的幻覺問題，並支援知識即時更新。"
    },
    {
        "id": "d04",
        "content": "向量資料庫（Vector Database）專門設計用於儲存和查詢高維度向量。常見的向量索引算法包括：HNSW（分層可導航小世界圖）適合高速近似搜索；IVF（倒排文件索引）適合大規模資料；Flat 索引提供精確搜索但速度較慢。Pinecone、Weaviate、Qdrant 是主流的向量資料庫服務。"
    },
    {
        "id": "d05",
        "content": "Transformer 架構於 2017 年由 Google 在論文『Attention Is All You Need』中提出。Transformer 的核心機制是自注意力（Self-Attention），允許模型動態關注輸入序列的不同部分。Transformer 包含編碼器（Encoder）和解碼器（Decoder）兩個元件，分別用於理解和生成。"
    },
    {
        "id": "d06",
        "content": "LLM 微調（Fine-tuning）是將預訓練模型針對特定任務進行進一步訓練的技術。主要方法包括：全參數微調（Full Fine-tuning）更新所有參數但成本高；LoRA（低秩適應）只更新少量參數，大幅降低訓練成本；指令微調（Instruction Fine-tuning）讓模型學習遵從特定格式的指令。"
    },
    {
        "id": "d07",
        "content": "Embedding 模型將文字轉換為稠密向量（Dense Vector）。OpenAI 的 text-embedding-3-small 模型輸出 1536 維向量，text-embedding-3-large 輸出 3072 維。Google 的 text-embedding-004 支援 task_type 參數，可以針對檢索（RETRIEVAL_DOCUMENT）或問題（RETRIEVAL_QUERY）分別優化。"
    },
    {
        "id": "d08",
        "content": "BM25（Best Match 25）是資訊檢索中的經典排名算法，是 TF-IDF 的改進版本。BM25 考慮了詞頻飽和效應（避免一個詞出現太多次被過度加權）和文件長度正規化（避免長文件佔優勢）。BM25 在傳統資訊檢索任務中仍然是非常強的基準線，特別適合精確詞彙匹配。"
    },
]

print(f"已準備 {len(documents)} 份文件")
for doc in documents:
    print(f"  [{doc['id']}] {doc['content'][:60]}...")

## 4. 建立兩種搜索系統

In [ ]:
# ── 系統 A：向量搜索（Dense Retrieval）──

# 建立 ChromaDB collection
try:
    chroma_client.delete_collection("hybrid_demo")
except:
    pass

vector_collection = chroma_client.create_collection(
    name="hybrid_demo",
    metadata={"hnsw:space": "cosine"}
)

# 建立所有文件的 embedding
contents = [doc["content"] for doc in documents]
response = client.embeddings.create(
    model="text-embedding-3-small",
    input=contents
)
embeddings = [r.embedding for r in response.data]

vector_collection.add(
    ids=[doc["id"] for doc in documents],
    embeddings=embeddings,
    documents=contents
)

print("✅ 向量搜索系統已建立")

# ── 系統 B：BM25 搜索（Sparse Retrieval）──

def tokenize_zh(text: str) -> list[str]:
    """簡單的中文字元分詞（實際生產用 jieba 等分詞工具）"""
    # 移除標點，按字元分割（適合示範）
    import re
    text = re.sub(r'[（）「」《》【】\(\)\[\]、，。！？]', ' ', text)
    # 保留英文字母和數字作為完整 token
    tokens = re.findall(r'[a-zA-Z0-9\-\.]+|[\u4e00-\u9fff]', text)
    return [t.lower() for t in tokens]

# 建立 BM25 索引
tokenized_corpus = [tokenize_zh(doc["content"]) for doc in documents]
bm25 = BM25Okapi(tokenized_corpus)

print("✅ BM25 搜索系統已建立")

# 示範：看看 tokenize 後的效果
sample = tokenize_zh(documents[0]["content"])
print(f"\n文件 d01 的 tokens（前 20 個）：")
print(sample[:20])

## 5. 單獨測試兩種搜索

先看看各自的優缺點。

In [ ]:
def vector_search(query: str, n_results: int = 5) -> list[dict]:
    """向量搜索：回傳 [{'id', 'score', 'content'}]"""
    query_embedding = client.embeddings.create(
        model="text-embedding-3-small",
        input=query
    ).data[0].embedding
    
    results = vector_collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        include=["documents", "distances"]
    )
    
    return [
        {
            "id": results["ids"][0][i],
            "score": 1 - results["distances"][0][i],  # 轉為相似度
            "content": results["documents"][0][i]
        }
        for i in range(len(results["ids"][0]))
    ]


def bm25_search(query: str, n_results: int = 5) -> list[dict]:
    """BM25 搜索：回傳 [{'id', 'score', 'content'}]"""
    query_tokens = tokenize_zh(query)
    scores = bm25.get_scores(query_tokens)
    
    # 取前 n_results 個
    top_indices = np.argsort(scores)[::-1][:n_results]
    
    return [
        {
            "id": documents[idx]["id"],
            "score": float(scores[idx]),
            "content": documents[idx]["content"]
        }
        for idx in top_indices
    ]


def print_results(results: list[dict], method: str, n: int = 3):
    print(f"\n[{method}] 前 {n} 名：")
    for i, r in enumerate(results[:n]):
        print(f"  {i+1}. [{r['id']}] 分數={r['score']:.4f} | {r['content'][:60]}...")


# 測試案例 1：語意查詢（向量搜索應該表現更好）
query1 = "如何讓語言模型更好地理解上下文語意？"
print(f"查詢 1（語意）：'{query1}'")
print_results(vector_search(query1), "向量搜索")
print_results(bm25_search(query1), "BM25")

In [ ]:
# 測試案例 2：精確術語查詢（BM25 應該表現更好）
query2 = "BM25 和 TF-IDF 的關係"
print(f"查詢 2（精確術語）：'{query2}'")
print_results(vector_search(query2), "向量搜索")
print_results(bm25_search(query2), "BM25")

print("\n→ 注意：BM25 對 'BM25' 這個精確關鍵字的匹配更準確！")

## 6. 混合搜索：Reciprocal Rank Fusion（RRF）

**RRF 算法**：不直接融合分數（因為兩種系統的分數範圍不同），而是融合**排名**。

$$\text{RRF\_Score}(d) = \sum_{i} \frac{1}{k + \text{rank}_i(d)}$$

- $k$：調節常數（通常設為 60），防止排名太靠前的文件過度主導
- $\text{rank}_i(d)$：文件 $d$ 在第 $i$ 個搜索系統中的排名

**直覺：** 在多個搜索系統中都排名靠前的文件，獲得更高的綜合分數。

In [ ]:
def reciprocal_rank_fusion(
    result_lists: list[list[dict]],
    k: int = 60
) -> list[dict]:
    """
    Reciprocal Rank Fusion - 融合多個排名列表
    Args:
        result_lists: 多個搜索結果列表，每個元素是 {'id', 'score', 'content'}
        k: RRF 常數（通常 60）
    Returns:
        融合後的排名列表
    """
    # 收集所有文件的 RRF 分數
    rrf_scores = {}  # doc_id -> RRF score
    doc_content = {}  # doc_id -> content
    
    for result_list in result_lists:
        for rank, doc in enumerate(result_list):
            doc_id = doc["id"]
            
            # RRF 公式：1 / (k + rank)
            # rank 從 0 開始，所以 rank+1 是實際排名
            rrf_score = 1 / (k + rank + 1)
            
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + rrf_score
            doc_content[doc_id] = doc["content"]
    
    # 按 RRF 分數排序
    sorted_results = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    
    return [
        {
            "id": doc_id,
            "score": score,
            "content": doc_content[doc_id]
        }
        for doc_id, score in sorted_results
    ]


def hybrid_search(query: str, n_results: int = 5, verbose: bool = True) -> list[dict]:
    """混合搜索 = 向量搜索 + BM25 + RRF 融合"""
    # 從兩個系統各取候選結果
    vec_results = vector_search(query, n_results=n_results)
    bm25_results = bm25_search(query, n_results=n_results)
    
    if verbose:
        print(f"向量搜索找到 {len(vec_results)} 個候選")
        print(f"BM25 搜索找到 {len(bm25_results)} 個候選")
    
    # RRF 融合
    fused_results = reciprocal_rank_fusion([vec_results, bm25_results])
    
    return fused_results


# RRF 分數示意圖
print("RRF 分數計算示例（k=60）：")
print("-" * 40)
for rank in [0, 1, 2, 4, 9]:
    score = 1 / (60 + rank + 1)
    print(f"  排名 {rank+1}: RRF = 1/{60+rank+1} = {score:.6f}")
print("\n→ 排名越靠前，RRF 分數越高；但差距隨排名增加而縮小")

In [ ]:
# 測試混合搜索
print("=" * 60)
print("測試混合搜索 vs 單一搜索")
print("=" * 60)

test_queries = [
    "BM25 的工作原理",
    "如何讓語言模型學會新知識？",
    "GPT-4 的上下文窗口有多長？",
]

for query in test_queries:
    print(f"\n查詢：'{query}'")
    print("-" * 50)
    
    vec_top = vector_search(query, n_results=3)[0]
    bm25_top = bm25_search(query, n_results=3)[0]
    hybrid_top = hybrid_search(query, n_results=5, verbose=False)[0]
    
    print(f"向量搜索 Top1: [{vec_top['id']}] {vec_top['content'][:70]}...")
    print(f"BM25   Top1: [{bm25_top['id']}] {bm25_top['content'][:70]}...")
    print(f"混合搜索 Top1: [{hybrid_top['id']}] {hybrid_top['content'][:70]}...")

## 7. 重排序（Reranking）

### 兩階段檢索架構

```
查詢
 ↓
[第一階段] Retrieval（向量/BM25/混合）
 → 快速篩選出 Top-50 候選文件
 → 使用 Bi-Encoder（速度優先）
 ↓
[第二階段] Reranking（重排序）
 → 對 Top-50 做精確排名，取 Top-5
 → 使用 Cross-Encoder（精確度優先）
 ↓
最終 Top-K 文件送入 LLM
```

### Bi-Encoder vs Cross-Encoder

| | Bi-Encoder | Cross-Encoder |
|--|--|--|
| **輸入** | 問題和文件分開編碼 | 問題 + 文件一起輸入 |
| **速度** | ✅ 快（可預計算） | ❌ 慢（每對都要計算） |
| **精確度** | ❌ 較低 | ✅ 較高（能看到完整交互） |
| **用途** | 第一階段檢索 | 第二階段重排序 |

In [ ]:
# 方法 A：用 LLM 作為 Reranker（簡單但有效）
# 這在沒有 Cross-Encoder 模型時是個好選擇

def llm_rerank(query: str, candidates: list[dict], top_k: int = 3) -> list[dict]:
    """
    使用 LLM 對候選文件重新排序
    讓 LLM 評估每個文件對查詢的相關性
    """
    # 建構評估 prompt
    candidates_text = ""
    for i, doc in enumerate(candidates):
        candidates_text += f"\n文件 {i+1}：\n{doc['content']}\n"
    
    prompt = f"""你是一個資訊檢索評估專家。請評估以下每份文件與查詢的相關性。

查詢：{query}

候選文件：{candidates_text}

請按照與查詢的相關性從高到低，輸出文件編號排名。
格式：只輸出數字，用逗號分隔，例如：3,1,2
只輸出數字排名，不要任何解釋。"""
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    
    # 解析排名
    ranking_str = response.choices[0].message.content.strip()
    try:
        rankings = [int(x.strip()) - 1 for x in ranking_str.split(",")]
        # 按照 LLM 給出的排名重新排序
        reranked = [candidates[i] for i in rankings if i < len(candidates)]
        return reranked[:top_k]
    except:
        # 解析失敗時直接回傳原始順序
        return candidates[:top_k]


# 測試 LLM Reranker
test_query = "向量資料庫如何儲存和搜索高維向量？"

# Step 1：混合搜索取候選
candidates = hybrid_search(test_query, n_results=5, verbose=False)
print(f"查詢：'{test_query}'")
print("\n[混合搜索] 初始排名（前 5 個）：")
for i, doc in enumerate(candidates):
    print(f"  {i+1}. [{doc['id']}] RRF={doc['score']:.6f} | {doc['content'][:60]}...")

# Step 2：LLM Reranking
print("\n🔄 執行 LLM Reranking...")
reranked = llm_rerank(test_query, candidates, top_k=3)

print("\n[Reranking 後] 最終 Top-3：")
for i, doc in enumerate(reranked):
    print(f"  {i+1}. [{doc['id']}] {doc['content'][:80]}...")

In [ ]:
# 方法 B：用 Cross-Encoder 模型 Reranking（效果更好）
# 這需要 sentence-transformers 套件

try:
    from sentence_transformers.cross_encoder import CrossEncoder
    
    # 使用多語言 Cross-Encoder
    # 'cross-encoder/ms-marco-MiniLM-L-6-v2' 適合英文
    # 'BAAI/bge-reranker-base' 支援中文
    print("載入 Cross-Encoder 模型 (BAAI/bge-reranker-base)...")
    cross_encoder = CrossEncoder("BAAI/bge-reranker-base")
    
    def cross_encoder_rerank(query: str, candidates: list[dict], top_k: int = 3) -> list[dict]:
        """使用 Cross-Encoder 重排序"""
        # 建構 (query, document) 對
        pairs = [(query, doc["content"]) for doc in candidates]
        
        # 計算相關性分數
        scores = cross_encoder.predict(pairs)
        
        # 按分數排序
        sorted_indices = np.argsort(scores)[::-1]
        
        reranked = []
        for idx in sorted_indices[:top_k]:
            doc = candidates[idx].copy()
            doc["rerank_score"] = float(scores[idx])
            reranked.append(doc)
        
        return reranked
    
    # 測試
    candidates = hybrid_search(test_query, n_results=5, verbose=False)
    reranked_ce = cross_encoder_rerank(test_query, candidates)
    
    print(f"\n查詢：'{test_query}'")
    print("\n[Cross-Encoder Reranking 後] 最終 Top-3：")
    for i, doc in enumerate(reranked_ce):
        print(f"  {i+1}. [{doc['id']}] score={doc['rerank_score']:.4f} | {doc['content'][:60]}...")

except ImportError:
    print("sentence-transformers 未安裝，跳過 Cross-Encoder 示範")
    print("安裝方式：uv add sentence-transformers")
except Exception as e:
    print(f"Cross-Encoder 載入失敗（需要下載模型）: {e}")
    print("跳過此示範，使用 LLM Reranker 即可達到類似效果")

## 8. 完整進階 RAG Pipeline

整合混合搜索 + Reranking + 生成，建立完整的進階 RAG 系統。

In [ ]:
def advanced_rag(
    query: str,
    n_candidates: int = 5,   # 第一階段候選數量
    n_final: int = 3,        # 最終送入 LLM 的文件數
    use_rerank: bool = True
) -> dict:
    """完整進階 RAG Pipeline"""
    
    print(f"🔍 查詢：{query}")
    print("=" * 60)
    
    # Stage 1：混合搜索
    print("[Stage 1] 混合搜索...")
    candidates = hybrid_search(query, n_results=n_candidates, verbose=False)
    print(f"  → 找到 {len(candidates)} 個候選文件")
    
    # Stage 2：Reranking
    if use_rerank:
        print("[Stage 2] LLM Reranking...")
        final_docs = llm_rerank(query, candidates, top_k=n_final)
    else:
        final_docs = candidates[:n_final]
    
    print(f"  → 精選 {len(final_docs)} 個最相關文件")
    for i, doc in enumerate(final_docs):
        print(f"     {i+1}. [{doc['id']}] {doc['content'][:60]}...")
    
    # Stage 3：生成
    print("[Stage 3] 生成回答...")
    
    context_parts = []
    for i, doc in enumerate(final_docs):
        context_parts.append(f"[參考資料 {i+1}]\n{doc['content']}")
    context = "\n\n".join(context_parts)
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "你是專業知識問答助理。根據參考資料回答問題，確保回答準確且有依據。"
            },
            {
                "role": "user",
                "content": f"參考資料：\n{context}\n\n問題：{query}"
            }
        ],
        temperature=0.1
    )
    
    answer = response.choices[0].message.content
    
    print("\n💬 最終回答：")
    print("-" * 60)
    print(answer)
    
    return {
        "query": query,
        "retrieved_docs": final_docs,
        "answer": answer
    }

# 完整測試
result = advanced_rag(
    "BERT 的預訓練方法和 GPT 有什麼本質差異？",
    n_candidates=5,
    n_final=3
)

In [ ]:
# 再測試一個精確術語查詢
print("\n" + "=" * 60)
result2 = advanced_rag(
    "BM25 為什麼比普通 TF-IDF 更好？",
    n_candidates=5,
    n_final=2
)

## 9. 生產系統考量

### 延遲預算（Latency Budget）

```
純向量搜索：     ~50ms
BM25 搜索：      ~5ms
混合搜索（RRF）： ~60ms
Cross-Encoder：  ~200-500ms（取決於候選數量）
LLM Reranker：   ~500-2000ms
LLM 生成：       ~1000-5000ms

完整 Pipeline P95 目標：< 5 秒
```

### 設計決策

| 場景 | 推薦方案 |
|------|----------|
| 延遲要求 < 1s | 純向量搜索，不做 Reranking |
| 延遲要求 < 3s | 混合搜索 + 輕量 Cross-Encoder |
| 品質優先 | 混合搜索 + LLM Reranker（非同步） |
| 成本優先 | 向量搜索 + BM25（交替使用） |

### 面試關鍵問答
> **面試官：** 為什麼要做混合搜索？向量搜索不夠嗎？
>
> **好答案：** 「向量搜索在語意理解上很強，但對精確關鍵字匹配（如專有名詞、版本號）可能失準。BM25 反過來對精確詞彙匹配更可靠但缺乏語意理解。混合搜索取兩者的聯集，用 RRF 融合排名而不是分數（因為兩個系統的分數範圍不同）。在生產系統中，混合搜索通常能比純向量搜索多 10-20% 的召回率。」

## 小結

1. ✅ **向量搜索的盲點**：精確術語、專有名詞匹配較弱
2. ✅ **BM25 的互補優勢**：關鍵字精確匹配更可靠
3. ✅ **混合搜索（RRF）**：不融合分數，融合排名，兩者優點兼得
4. ✅ **兩階段架構**：第一階段快速篩選（Bi-Encoder），第二階段精確排名（Cross-Encoder）
5. ✅ **Reranking**：在延遲和品質之間的關鍵權衡

### 下一步
- **NB04**：如何量化評估 RAG 系統的好壞？RAGAS 評估框架